In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import torch

from sentence_transformers import SentenceTransformer,InputExample,losses,util

from torch.utils.data import DataLoader

from src.metric import *

In [ ]:
with open('../data/cleaned/clean_train_df.dill','rb') as f:
    train_df=dill.load(f)
    
with open('../data/cleaned/clean_val_df.dill','rb') as f:
    val_df=dill.load(f)
        
with open('../data/cleaned/clean_test_df.dill','rb') as f:
    test_df=dill.load(f)
    


In [ ]:
device="cuda" if torch.cuda.is_available() else "cpu"
print(device)

In [ ]:
model_path='../models/bi_encoder'
os.makedirs(model_path,exist_ok=True)

In [ ]:
bi_encoder= SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2',device=device)

In [ ]:
label_to_score = {0: 0.0, 1: 0.6, 2: 1.0}

In [ ]:
bi_train_examples=[
    InputExample(texts=[j,r],label=float(label_to_score[l]))
    for r,j,l in zip(train_df['resume_text'],train_df['job_description_text'],train_df['label'])
]

bi_train_dataloader=DataLoader(bi_train_examples,shuffle=True,batch_size=32)

bi_train_loss=losses.CoSENTLoss(bi_encoder)

In [ ]:
epochs = 4
best_score = float('-inf')

print("\tTraining Phase")
for epoch in range(1, epochs + 1):
    print(f"Epoch: {epoch}----------")

    bi_encoder.fit(
        train_objectives=[(bi_train_dataloader, bi_train_loss)],
        epochs=1,
        warmup_steps=int(len(bi_train_dataloader) * epochs * 0.1),
        show_progress_bar=True
    )

    val_resume_emb=bi_encoder.encode(
        val_df['resume_text'].tolist(),
        batch_size=32,convert_to_tensor=True,show_progress_bar=False
    )
    val_jd_emb =bi_encoder.encode(
        val_df['job_description_text'].tolist(),
        batch_size=32,convert_to_tensor=True,show_progress_bar=False
    )

    scores = torch.cosine_similarity(val_resume_emb, val_jd_emb).cpu().numpy()

    metrics = model_evaluation(scores, val_df, 'job_description_text')
    print(f"Spearman Score:{metrics['spearman_score']}")
    print(f"Top 3 score:{metrics['topk_score']}")
    print(f"NDGC Score:{metrics['ndcg_val']}")
    print(f"MAP Score:{metrics['mrr_score']}")
    print(f"MRR Score:{metrics['map_score']}")

    final_score = (0.6*metrics['ndcg_val'] +
                   0.3*metrics['map_score'] +
                   0.1*(metrics['mrr_score'] + metrics['topk_score']))

    if final_score > best_score:
        best_score=final_score
        bi_encoder.save(model_path)
        

print("\tInference Phase")
val_resume_emb=bi_encoder.encode(val_df['resume_text'].tolist(),
                              batch_size=32,convert_to_tensor=True,
                              show_progress_bar=True)
val_jd_emb = bi_encoder.encode(val_df['job_description_text'].tolist(),
                              batch_size=32,convert_to_tensor=True,
                              show_progress_bar=True)

scores = torch.cosine_similarity(val_resume_emb, val_jd_emb).cpu().numpy()

In [ ]:
cos_sim=torch.cosine_similarity(scores,val_jd_emb)

In [ ]:
model_evaluation(scores,val_df,'job_description_text')